# 08 KNN With Basic Image Features

This notebook trains a basic `KNeighborsClassifier`, then tunes only KNN using grouped cross-validation on the training split. KNN is a similarity-based supervised classifier: it predicts a crop from nearby training examples in scaled feature space.

## 1. Project setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

## 2. Imports

In [ ]:
from src.extract_basic_features import BASIC_FEATURE_COLUMNS
from src.train_logistic_regression import FEATURES_CSV, load_features
from src.train_knn import (
    BASIC_SCORES_CSV,
    TUNED_SCORES_CSV,
    BASIC_CONFUSION_MATRIX_FIGURE,
    TUNED_CONFUSION_MATRIX_FIGURE,
    build_comparison_table,
    print_overfitting_summary,
    train_knn,
)

## 3. Paths and configuration

In [ ]:
print(f"Basic features CSV: {FEATURES_CSV}")
print(f"Basic scores CSV: {BASIC_SCORES_CSV}")
print(f"Tuned scores CSV: {TUNED_SCORES_CSV}")
print(f"Basic confusion matrix figure: {BASIC_CONFUSION_MATRIX_FIGURE}")
print(f"Tuned confusion matrix figure: {TUNED_CONFUSION_MATRIX_FIGURE}")

## 4. Load metadata

In [ ]:
features = load_features(FEATURES_CSV)
features.head()

## 5. Sanity checks

In [ ]:
print(f"Rows: {len(features)}")
print("Label counts:")
print(features["label"].value_counts().sort_index())
print(f"Image feature columns: {len(BASIC_FEATURE_COLUMNS)}")
print(BASIC_FEATURE_COLUMNS)
assert set(BASIC_FEATURE_COLUMNS).issubset(features.columns)
assert features["area_group"].notna().all()

## 6. Main analysis

Only image-derived feature columns are used as predictors. Metadata columns are used for labels, auditing, and grouped splitting, not as model inputs. KNN is distance-based, so `StandardScaler` is included inside the model pipeline.

In [ ]:
(
    basic_scores,
    basic_report,
    basic_matrix,
    basic_figure_path,
    tuned_scores,
    tuned_report,
    tuned_matrix,
    tuned_figure_path,
    y_train,
    y_test,
    search,
) = train_knn(features)

basic_scores

## 7. Results/figures

Balanced accuracy and macro F1 matter more than raw accuracy because the dataset has many more ordered images than disordered images.

In [ ]:
basic_scores.to_csv(BASIC_SCORES_CSV, index=False)
tuned_scores.to_csv(TUNED_SCORES_CSV, index=False)

print("Basic KNN per-class precision/recall/F1:")
print(basic_report)
print("Tuned KNN per-class precision/recall/F1:")
print(tuned_report)

print_overfitting_summary(basic_scores, "Basic KNN")
print_overfitting_summary(tuned_scores, "Tuned KNN")

tuned_row = tuned_scores.iloc[0]
print("\nTuning summary:")
print(f"Best parameters: {search.best_params_}")
print(f"Best CV macro F1: {search.best_score_:.4f}")
print(f"Final test macro F1: {tuned_row['macro_f1']:.4f}")
print(f"Final test balanced accuracy: {tuned_row['test_balanced_accuracy']:.4f}")
print(f"Disordered recall: {tuned_row['disordered_recall']:.4f}")

print("\nModel comparison:")
print(build_comparison_table(basic_scores, tuned_scores))

print(f"\nSaved basic scores to {BASIC_SCORES_CSV}")
print(f"Saved tuned scores to {TUNED_SCORES_CSV}")
print(f"Saved basic confusion matrix figure to {basic_figure_path}")
print(f"Saved tuned confusion matrix figure to {tuned_figure_path}")

tuned_scores

## 8. Notes for report

- KNN is a similarity-based classifier.
- It does not learn explicit weights like logistic regression or split rules like decision trees.
- It predicts based on nearby examples in scaled feature space.
- Scaling is required because distance-based models are sensitive to feature magnitude.
- KNN may struggle if the feature space is noisy or high-dimensional.
- This step does not add SVC, HOG, graph features, augmentation, or CNNs.